# Урок 3.1. RAG с нуля: первый рабочий пайплайн

**Модуль 3 «RAG: Retrieval-Augmented Generation на практике» · Курс «LLM для Python-разработчиков»**

Этот урок — момент, ради которого строились два предыдущих модуля. В модуле 1 вы научились разговаривать с LLM; в модуле 2 — находить нужные фрагменты в базе знаний. Сегодня соединяем половины: **поиск находит контекст → LLM пишет по нему ответ**. Это и есть RAG — и вы соберёте его на чистом Python, без единого фреймворка, понимая каждую строчку.

## 🎯 Цели урока

После этого урока вы сможете:

- объяснить, **зачем существует RAG** и почему это главная архитектура LLM-приложений;
- собрать полный конвейер: **индексация → поиск → промпт → генерация → источники**;
- писать **RAG-промпт** с grounding-инструкциями и нумерованными источниками;
- поставить **чистый эксперимент**, доказывающий, что модель отвечает из документов, а не из своих весов;
- увидеть **два слабых места** первой версии — и знать, какие уроки модуля их лечат.

## План

1. Зачем RAG: три способа дать LLM знания
2. Шаг 1 — ретривер (всё из модуля 2)
3. Шаг 2 — RAG-промпт (всё из модуля 1)
4. Шаг 3 — генерация и первый ответ
5. Чистый эксперимент: RAG против голой модели
6. Источники и пределы первой версии

⏱ **Время:** ~75–90 минут.

> 💡 Понадобится Ollama с `qwen2.5:3b` (модуль 1). Ретривер в этом уроке — лексический (работает сразу); семантический апгрейд на e5 — в guarded-секции, как обычно.

## 1. Зачем RAG: три способа дать LLM знания

Спросите `qwen2.5:3b` про правила отпусков в «Векторике» — и получите либо честное «не знаю», либо уверенную выдумку (урок 1.1: cutoff и галлюцинации). Модель не видела ваших внутренних документов и не могла. Как дать ей эти знания?

| способ | как | почему (не) работает |
|---|---|---|
| **всё в промпт** | вставить всю базу в контекст | контекст не резиновый (урок 1.1), а каждый токен стоит денег и латентности (урок 1.3); на 10 000 документов — невозможно |
| **fine-tuning** | дообучить модель на документах | дорого, медленно обновлять, а главное — дообучение меняет *поведение*, но плохо вбивает *факты* (урок 1.6); модель всё равно будет путаться |
| **RAG** | найти релевантные фрагменты и вставить **только их** | база обновляется без переобучения, источники проверяемы, платим токенами только за нужное |

**RAG (Retrieval-Augmented Generation)** — «генерация, усиленная поиском». Архитектура целиком — два конвейера, и оба вам знакомы:

```text
ОФЛАЙН (индексация - модуль 2):
  документы -> чанки + заголовки -> индекс (TF-IDF / эмбеддинги / БД)

ОНЛАЙН (на каждый вопрос):
  вопрос ----> ретривер: топ-k чанков      (модуль 2)
                    |
                    v
               RAG-промпт: инструкция + [1][2][3] контекст + вопрос   (модуль 1)
                    |
                    v
               LLM -> ответ с опорой на контекст + ссылки на источники
```

Ключевое слово урока — **grounding** («заземление»): ответ модели должен опираться на предоставленный контекст, а не на её собственные веса. Всё остальное — инженерия вокруг этого требования.

In [ ]:
import httpx

from utils import chunk_paragraphs, load_documents

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen2.5:3b"

docs = load_documents("data")
print(f"База знаний «Векторики»: {len(docs)} документов")

try:
    tags = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=3).json()
    installed = [m["name"] for m in tags.get("models") or []]
    OLLAMA_AVAILABLE = MODEL in installed
    if OLLAMA_AVAILABLE:
        print(f"Ollama на месте, модель {MODEL} готова — урок будет живым.")
    else:
        print(f"Ollama работает, но {MODEL} не скачана: ollama pull {MODEL}")
except (httpx.ConnectError, httpx.TimeoutException):
    OLLAMA_AVAILABLE = False
    print("Ollama не запущена (урок 1.5) — генеративные ячейки пропустятся.")

In [ ]:
def chat(messages: list, *, temperature: float = 0.2, num_predict: int = 400) -> str | None:
    """Вызов LLM из урока 1.2. Низкая температура - для фактических ответов."""
    if not OLLAMA_AVAILABLE:
        print("[пропущено: Ollama или модель недоступны]")
        return None
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL, "messages": messages, "stream": False,
        "options": {"temperature": temperature, "num_predict": num_predict},
    }, timeout=300)
    response.raise_for_status()
    return response.json()["message"]["content"]

## 2. Шаг 1 — ретривер

Ретривер у нас готов со времён модуля 2: чанкинг по абзацам, контекстные заголовки, TF-IDF. Собираем компактную версию (в уроке 2.7 то же самое было классом `LexicalBackend`).

Честная оговорка: лексический ретривер — **слабейшее звено** этого пайплайна («Упал прод» он не найдёт — модуль 2 это доказал). Мы берём его, потому что он работает без скачиваний, а механика RAG не зависит от типа ретривера: замена на e5 + Qdrant — это одна функция (guarded-секция в конце и весь урок 3.2).

In [ ]:
import math
import re

# --- индексация: документы -> чанки с заголовками (уроки 2.5-2.6) ---
chunks: list[dict] = []
for doc in docs:
    for i, piece in enumerate(chunk_paragraphs(doc.text, max_chars=500)):
        chunks.append({
            "chunk_id": f"{doc.doc_id}_{i}",
            "doc_id": doc.doc_id,
            "title": doc.title,
            "text": piece,
            "indexed_text": piece if i == 0 else f"{doc.title}. {piece}",
        })

def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

token_lists = [tokenize(c["indexed_text"]) for c in chunks]
df: dict[str, int] = {}
for tokens in token_lists:
    for word in set(tokens):
        df[word] = df.get(word, 0) + 1
idf = {word: math.log(len(chunks) / count) for word, count in df.items()}

def retrieve(question: str, top_k: int = 3) -> list[dict]:
    """Топ-k чанков по TF-IDF (урок 2.5). Возвращает чанки со score."""
    query_words = set(tokenize(question))
    scored = []
    for chunk, tokens in zip(chunks, token_lists):
        score = sum((tokens.count(w) / len(tokens)) * idf.get(w, 0.0)
                    for w in query_words)
        scored.append((score, chunk))
    scored.sort(key=lambda pair: -pair[0])
    return [{**chunk, "score": score} for score, chunk in scored[:top_k]]

print(f"Ретривер готов: {len(chunks)} чанков в индексе")

In [ ]:
# проверим находки перед тем, как отдавать их модели
for found in retrieve("Сколько компания доплачивает в первые дни болезни?"):
    print(f"[{found['score']:.3f}] {found['chunk_id']:<14} {found['text'][:80]}...")

## 3. Шаг 2 — RAG-промпт

Сердце RAG — промпт, склеивающий найденное с вопросом. Анатомия — сплошные приёмы модуля 1:

```text
СИСТЕМА:   роль + правила grounding                        (урок 1.2)
           "отвечай ТОЛЬКО по контексту"
           "нет ответа в контексте -> так и скажи"
           "ссылайся на источники по номерам"

КОНТЕКСТ:  [1] Больничные: <текст чанка>                   (нумерация - чтобы
           [2] Отпуск: <текст чанка>                        модель могла ссылаться,
           в разделителях <context>...</context>            разделители - урок 1.2)

ВОПРОС:    вопрос пользователя как есть
```

Две инструкции здесь несут всю тяжесть grounding'а. «Отвечай только по контексту» прижимает модель к документам. «Если ответа нет — скажи об этом» даёт ей **разрешение не знать**: без него модель, обученная быть полезной (урок 1.6 — SFT-манеры), предпочтёт выдумать, чем промолчать.

In [ ]:
SYSTEM_RAG = (
    "Ты — ассистент по внутренней базе знаний компании «Векторика».\n"
    "Правила:\n"
    "1. Отвечай ТОЛЬКО на основе фрагментов из блока <context>. "
    "Не используй никакие другие знания.\n"
    "2. Если в контексте нет ответа на вопрос — честно скажи: "
    "«В базе знаний нет ответа на этот вопрос».\n"
    "3. В конце ответа укажи использованные источники в виде [номер].\n"
    "4. Отвечай кратко и по делу, по-русски."
)

def build_prompt(question: str, found_chunks: list[dict]) -> list[dict]:
    """Собирает messages для LLM: система + контекст + вопрос."""
    context = "\n\n".join(
        f"[{i}] {chunk['title']}:\n{chunk['text']}"
        for i, chunk in enumerate(found_chunks, start=1)
    )
    user_message = (
        f"<context>\n{context}\n</context>\n\n"
        f"Вопрос: {question}"
    )
    return [{"role": "system", "content": SYSTEM_RAG},
            {"role": "user", "content": user_message}]

# посмотрим на промпт целиком - модель увидит именно это
demo_messages = build_prompt(
    "Сколько компания доплачивает в первые дни болезни?",
    retrieve("Сколько компания доплачивает в первые дни болезни?"),
)
print(demo_messages[1]["content"][:600], "...")

## 4. Шаг 3 — генерация. Весь RAG в пяти строках

Осталось соединить три шага. Обратите внимание, насколько это короткая функция — вся сложность уже разложена по кирпичикам модулей 1 и 2:

In [ ]:
def rag_answer(question: str, top_k: int = 3) -> dict:
    """Полный RAG-конвейер: поиск -> промпт -> генерация."""
    found = retrieve(question, top_k=top_k)
    messages = build_prompt(question, found)
    answer = chat(messages)
    return {"answer": answer, "sources": found}

result = rag_answer("Сколько компания доплачивает в первые дни болезни?")
if result["answer"]:
    print(result["answer"])
    print("\nИсточники:")
    for i, chunk in enumerate(result["sources"], start=1):
        print(f"  [{i}] {chunk['chunk_id']} — {chunk['title']}")

> **Пример ответа** (ваш может отличаться в формулировках):
>
> «Компания доплачивает до 100% оклада первые 10 дней болезни, если стаж работы в «Векторике» больше года [1]».

Числа — из документа, ссылка — на источник. Но постойте: может, модель и так это знала? Проверим по-настоящему.

## 5. Чистый эксперимент: RAG против голой модели

У нашей базы есть научное преимущество: **«Векторика» выдумана**. Её правило «трёх дней доверия», доплата «до 100% первые 10 дней при стаже от года», надбавка 15% за дежурство — этих фактов нет и не может быть в весах qwen2.5:3b, обученной до того, как эти документы были написаны. Значит, любой правильный ответ мог прийти только из контекста.

Зададим тот же вопрос голой модели — без единого документа:

In [ ]:
bare_answer = chat([
    {"role": "system", "content": "Ты — ассистент компании «Векторика». Отвечай кратко."},
    {"role": "user", "content": "Сколько компания доплачивает в первые дни болезни?"},
])
if bare_answer:
    print("=== БЕЗ RAG (голая модель) ===")
    print(bare_answer)
    print()
    print("=== С RAG (тот же вопрос) ===")
    print(result["answer"])

Типичный результат: голая модель либо отвечает уклончиво-обобщённо («зависит от политики компании...»), либо **уверенно называет выдуманные числа** — например, ссылается на общие нормы больничных из российского законодательства, которых в вопросе никто не спрашивал. Это не дефект qwen — это фундаментальное свойство LLM из урока 1.1: она продолжает текст правдоподобно, а не правдиво.

RAG-версия отвечает конкретными цифрами из документа и ссылается на источник. **Разница между этими двумя ответами — и есть весь смысл RAG.**

Пара слов о том, зачем в ответе `[1]`:

- **проверяемость**: пользователь может открыть документ и убедиться;
- **доверие**: ответ со ссылкой на «Больничные и отсутствие по болезни» убеждает сильнее анонимного утверждения;
- **отладка**: если ответ неверен, сразу видно, что сломалось — поиск (принёс не то) или генерация (переврала принесённое). Эта диагностика станет основой метрик в уроке 3.3.

## 6. Пределы первой версии

Наш RAG работает, но у него два слабых места, и второе мы сейчас продемонстрируем. Первое вы уже знаете: **лексический ретривер** («Упал прод» — мимо). Второе тоньше: ретривер **всегда возвращает топ-k**, даже если релевантных документов нет вообще. Что произойдёт, если спросить о том, чего в базе нет?

In [ ]:
off_topic = rag_answer("Какая столица у Франции?")
if off_topic["answer"]:
    print("Вопрос: Какая столица у Франции?\n")
    print(off_topic["answer"])
    print("\nЧто принёс ретривер:")
    for i, chunk in enumerate(off_topic["sources"], start=1):
        print(f"  [{i}] score={chunk['score']:.4f}  {chunk['chunk_id']}")

Посмотрите на score источников: они близки к нулю — ретривер честно сигналит «ничего похожего не нашёл», но всё равно отдаёт топ-3 наименее непохожего мусора. Дальше — лотерея, и оба исхода поучительны:

- модель **послушалась grounding-инструкции** и ответила «в базе знаний нет ответа» — отлично, промпт отработал;
- модель **ответила «Париж» из своих весов** — grounding сломался: инструкция «только по контексту» проиграла знанию, вшитому в веса (помните из урока 1.7 — инструкции слабее, чем хочется думать).

Надёжное решение — не молиться на промпт, а **не звать LLM вовсе**, когда лучший score ниже порога: нет релевантного контекста → сразу «не знаю», ноль токенов потрачено. Пороги, «я не знаю»-ответы и цитирование — целый урок 3.4. А в уроке 3.3 мы научимся *измерять*, как часто наш RAG галлюцинирует, — eval-дисциплина дойдёт и до генерации.

## 7. Апгрейд ретривера (для владельцев e5)

Вся прелесть разобранной архитектуры: чтобы заменить ретривер, меняется **одна функция** — `retrieve`. Ни промпт, ни генерация не узнают о замене:

In [ ]:
import numpy as np

EMB_MODEL_NAME = "intfloat/multilingual-e5-small"

emb_model = None
try:
    from sentence_transformers import SentenceTransformer
    from huggingface_hub import snapshot_download
    snapshot_download(EMB_MODEL_NAME, local_files_only=True)
    emb_model = SentenceTransformer(EMB_MODEL_NAME)
    print("Модель e5 загружена — собираем семантический RAG.")
except Exception:
    print("Модель e5 не в кэше (урок 2.1) — секция пропустится.")

if emb_model is not None and OLLAMA_AVAILABLE:
    chunk_vecs = emb_model.encode(
        [f"passage: {c['indexed_text']}" for c in chunks],
        normalize_embeddings=True)

    def retrieve_semantic(question: str, top_k: int = 3) -> list[dict]:
        q_vec = emb_model.encode([f"query: {question}"], normalize_embeddings=True)[0]
        scores = chunk_vecs @ q_vec
        order = np.argsort(scores)[::-1][:top_k]
        return [{**chunks[i], "score": float(scores[i])} for i in order]

    # вопрос, на котором лексика сдавалась весь модуль 2:
    question = "Упал прод, что делать в первую очередь?"
    found = retrieve_semantic(question)
    answer = chat(build_prompt(question, found))
    print(f"Вопрос: {question}\n")
    print(answer)
    print("\nИсточники:", [c["chunk_id"] for c in found])

Семантический ретривер закрывает «Упал прод» (находит документ про инциденты по смыслу), а RAG-обвязка не изменилась ни на символ. В уроке 3.2 этот конвейер переедет на Qdrant с асинхронностью и завернётся в FastAPI-эндпоинт — production-путь.

## ⚠️ Частые ошибки

1. **Проверять RAG на фактах, которые модель и так знает.** «Кто написал Войну и мир?» ответится из весов, и вы не узнаете, работает ли ваш поиск. Тестируйте на приватных/выдуманных фактах — у нас для этого вся «Векторика».
2. **Забыть разрешение «не знать».** Без инструкции «нет в контексте — так и скажи» модель заполнит пробел выдумкой: SFT приучил её быть полезной любой ценой (урок 1.6).
3. **Высокая температура для фактических ответов.** 0.7 хороша для чата, но RAG-ответ должен быть максимально детерминированным пересказом контекста: 0.0–0.3 (урок 1.2).
4. **Скармливать LLM мусор при пустом поиске.** Ретривер всегда вернёт топ-k. Нет порога — нет защиты: расход токенов и провокация галлюцинаций на каждый нерелевантный вопрос.
5. **Контекст без структуры.** Чанки, слепленные без номеров и заголовков, лишают модель возможности ссылаться, а вас — отлаживать. Нумерация `[i]` — три строчки кода и половина будущих метрик.
6. **Слишком много контекста «на всякий случай».** top_k=10 на простой вопрос — это деньги (урок 1.3), латентность (prefill! урок 1.5) и шум, в котором тонет нужный фрагмент (упражнение 1).

## 🏋️ Упражнения

### Упражнение 1 — сколько контекста достаточно? 🎛

Прогоните один и тот же вопрос с `top_k` = 1, 3 и 6. Сравните ответы: где не хватило информации, где появился шум? Посчитайте примерную длину промпта в токенах для каждого варианта (примерно символы/3 — урок 1.1) и сформулируйте цену top_k.

### Упражнение 2 — сломайте grounding 🔨

Найдите вопрос, на который база отвечает *похоже, но не точно* — например, спросите о том, что упоминается вскользь, но не регламентируется («Можно ли работать из-за границы?» — в документе про удалёнку есть про удалёнку, но, возможно, не про границу). Посмотрите, честно ли модель разделяет «есть в контексте» и «додумано». Это ручная версия того, что урок 3.3 будет мерить автоматически.

### Упражнение 3 — стриминг для нетерпеливых ⚡

Переделайте `rag_answer` в `rag_answer_stream`: ответ печатается токен за токеном (урок 1.4, `stream: True` в `/api/chat`). Источники выведите до начала генерации — пользователь видит, «откуда» будет ответ, пока модель пишет.

## 📝 Мини-квиз

**1. Почему для базы внутренних документов RAG правильнее fine-tuning'а?**

<details><summary>Ответ</summary>

Fine-tuning меняет поведение модели, но плохо вбивает факты и требует переобучения при каждом обновлении базы. RAG хранит знания снаружи: база обновилась — следующий запрос уже видит новые документы; источники проверяемы; платим токенами только за найденное. (Fine-tuning нужен для другого — стиль, формат, домен — модуль 6.)

</details>

**2. Что такое grounding и какие две инструкции промпта за него отвечают?**

<details><summary>Ответ</summary>

Grounding — требование опираться в ответе на предоставленный контекст, а не на веса модели. Инструкции: «отвечай только по контексту» и «если ответа нет в контексте — скажи об этом» (разрешение не знать). Но промпт — мягкая защита; жёсткая — порог релевантности до вызова LLM.

</details>

**3. Почему мы проверяли RAG на выдуманной «Векторике», а не на общеизвестных фактах?**

<details><summary>Ответ</summary>

Правильный ответ про общеизвестное мог прийти из весов модели — эксперимент не отличил бы работающий RAG от неработающего. Факты «Векторики» в весах отсутствуют, поэтому верный ответ доказывает: контекст дошёл и был использован.

</details>

**4. Ретривер вернул топ-3 со score около нуля. Что должно произойти в зрелом RAG?**

<details><summary>Ответ</summary>

LLM вообще не вызывается: порог релевантности отсекает пустой поиск, пользователь получает честное «не знаю», система экономит токены и не рискует галлюцинацией. Полагаться только на промпт-инструкцию — лотерея (урок 3.4 — про пороги).

</details>

**5. По ответу с ошибкой как понять, виноват поиск или генерация?**

<details><summary>Ответ</summary>

Смотрим на источники (они возвращаются вместе с ответом): если нужного фрагмента нет среди найденных — сломан retrieval; если фрагмент есть, а ответ ему противоречит — сломана генерация (модель переврала контекст). Это разделение — фундамент метрик RAG в уроке 3.3.

</details>

## ✅ Решения упражнений

### Решение 1 — сколько контекста достаточно?

In [ ]:
question = "Как оформить отпуск?"

for top_k in (1, 3, 6):
    found = retrieve(question, top_k=top_k)
    prompt_chars = len(build_prompt(question, found)[1]["content"])
    print(f"=== top_k={top_k}: ~{prompt_chars // 3} токенов промпта ===")
    answer = chat(build_prompt(question, found), num_predict=200)
    if answer:
        print(answer[:400])
    print()

Типичная картина: `top_k=1` рискует упустить часть ответа (заявка — в одном чанке, сроки — в другом); `top_k=3` — баланс; `top_k=6` утраивает цену промпта, добавляя чанки про больничные и VPN, которые модель должна проигнорировать (а маленькая модель может и не проигнорировать). Размер контекста — та же ручка «полнота ↔ цена ↔ шум», что и везде в курсе, и настраивается она eval'ом (урок 3.3), а не интуицией.

### Решение 2 — сломайте grounding

In [ ]:
tricky = rag_answer("Можно ли работать удалённо из другой страны?")
if tricky["answer"]:
    print(tricky["answer"])
    print("\nЧто было в контексте:")
    for i, chunk in enumerate(tricky["sources"], start=1):
        print(f"  [{i}] {chunk['chunk_id']}: {chunk['text'][:100]}...")

In [ ]:
found = retrieve_semantic("Можно ли работать удалённо из другой страны?")

In [ ]:
print("=== Семантический поиск (e5) ===")
for chunk in found:
    print(f"[{chunk['score']:.3f}] {chunk['chunk_id']:<14} {chunk['text'][:80]}...")

In [ ]:
answer = chat(build_prompt("Можно ли работать удалённо из другой страны?", found))

In [ ]:
answer

В нашем прогоне упражнение приземлилось неожиданно — и поучительнее задуманного. Посмотрите на источники: ретривер **вообще не принёс документ про удалёнку!** Для лексического поиска «работать удалённо» и «удалёнка» — разные слова (морфология — старый враг из модуля 2), и в контекст уехали больничные с тестированием. Модель повела себя образцово — честно отказалась. Это живая иллюстрация квиз-вопроса №5: по источникам мгновенно видно, что сломался *поиск*, а не генерация.

Владельцам e5: повторите с `retrieve_semantic` — документ про удалёнку найдётся, и вот тогда сверьте ответ с его текстом глазами: сказала ли модель только то, что там написано, или дорисовала правдоподобное («нужно согласование с юристами» — а этого в документе нет)? Такое смешение источника с додумкой — самый коварный вид галлюцинации: ответ *выглядит* заземлённым. В уроке 3.3 будем ловить его автоматически — предложение за предложением.

### Решение 3 — стриминг для нетерпеливых

In [ ]:
import json as jsonlib

def rag_answer_stream(question: str, top_k: int = 3) -> None:
    """RAG со стримингом: источники сразу, ответ - токен за токеном (урок 1.4)."""
    if not OLLAMA_AVAILABLE:
        print("[пропущено: Ollama или модель недоступны]")
        return
    found = retrieve(question, top_k=top_k)
    print("Источники:", ", ".join(c["chunk_id"] for c in found), "\n")

    with httpx.stream("POST", f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL,
        "messages": build_prompt(question, found),
        "options": {"temperature": 0.2, "num_predict": 300},
        "stream": True,
    }, timeout=300) as response:
        for line in response.iter_lines():
            if not line:
                continue
            piece = jsonlib.loads(line)
            print(piece.get("message", {}).get("content", ""), end="", flush=True)
    print()

rag_answer_stream("Сколько дней отпуска положено в году? Дай полный ответ.")

Источники печатаются мгновенно (поиск — миллисекунды), ответ течёт токенами — TTFT вместо ожидания полного ответа (урок 1.4 про то, почему это решает UX). В FastAPI-версии урока 3.2 этот же приём станет server-sent events.

## 🎓 Итоги

- **RAG = ретривер (модуль 2) + промпт и генерация (модуль 1).** Сердце системы — `rag_answer` на пять строк; вся сложность разложена по знакомым кирпичикам.
- **RAG-промпт** = роль + grounding-правила («только по контексту», «разреши себе не знать») + нумерованный контекст в разделителях + вопрос.
- **Чистый эксперимент** — сравнение с голой моделью на выдуманных фактах: RAG отвечает цифрами из документов, голая модель — правдоподобной пустотой.
- **Источники** в ответе — проверяемость для пользователя и диагностика для вас: ошибка поиска и ошибка генерации лечатся по-разному.
- Два известных слабых места: лексический ретривер (лечится e5/Qdrant — урок 3.2) и отсутствие порога релевантности (мусор в контекст — урок 3.4).

**Дальше — урок 3.2 «RAG в продакшене»:** переезд на Qdrant, асинхронный клиент и FastAPI-эндпоинт — из ноутбука в сервис.

## 📚 Что почитать

- [Оригинальная статья RAG (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401) — откуда пошло название
- [Anthropic: Retrieval Augmented Generation](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/long-context-tips) — практика длинного контекста и цитирования
- [Ollama API](https://github.com/ollama/ollama/blob/main/docs/api.md) — стриминг и параметры, которые мы использовали
- [RAG Survey (2024)](https://arxiv.org/abs/2312.10997) — большой обзор вариаций RAG: пригодится как карта модуля 4